# 6. Visualise PyNNLF SA BESS Results

This notebook compares the three SA BESS datasets used in the PyNNLF experiment. The focus is the change in load shape, distribution, and model performance between underlying load, net load with PV, and net load with PV plus battery.

Dataset provenance:
- Source: Solar Analytics CICCADA BESS aggregate processed by the publication workflow.
- Sample: 100 households.
- Period: 2024-02-26 00:00 to 2025-02-24 23:30 fixed AEST.
- Length: 17,520 half-hour rows per dataset, covering 365 days.
- Signal profile: `current_polarity_adjusted`.
- No missing values are expected in the exported 30-minute datasets.

## Setup

Load the exported SA BESS datasets and PyNNLF result tables.

In [ ]:
from pathlib import Path
import sys

import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.patches import Patch

def find_publication_project(start: Path) -> Path:
    """Find publication/journal_article_1 from this moved notebook folder."""
    for candidate in [start.resolve(), *start.resolve().parents]:
        if (candidate / 'scripts' / 'process_pynnlf_output.py').exists() and (candidate / 'data').exists():
            return candidate
    raise FileNotFoundError('Could not find publication/journal_article_1 from the current working directory.')


PROJECT_DIR = find_publication_project(Path.cwd())
DATA_DIR = PROJECT_DIR / 'data'
RESULTS_DIR = PROJECT_DIR / 'results' / '00_data_exploration_and_processing'
FIGURES_DIR = RESULTS_DIR / 'figures' / '01_sa_bess_with_100hh'
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

print(f'Publication project: {PROJECT_DIR}')
print(f'Figure output folder: {FIGURES_DIR}')

N_HOUSEHOLDS = 100
N_DAYS = 365
SIGNAL_PROFILE = 'current_polarity_adjusted'
DATASET_ORDER = ['underlying_load', 'net_load_with_pv', 'net_load_with_pv_battery']
DATASET_SPECS = {
    'underlying_load': {
        'filename': 'ds16_sa_bess_underlying_load_30min.csv',
        'label': 'Underlying load',
        'definition': 'Aggregate underlying household demand before PV and battery effects.',
    },
    'net_load_with_pv': {
        'filename': 'ds17_sa_bess_net_load_with_pv_30min.csv',
        'label': 'Net load with PV',
        'definition': 'Aggregate load after PV generation, before battery operation.',
    },
    'net_load_with_pv_battery': {
        'filename': 'ds18_sa_bess_net_load_with_pv_battery_30min.csv',
        'label': 'Net load with PV and battery',
        'definition': 'Aggregate load after PV generation and battery operation.',
    },
}
DATASET_LABELS = {key: spec['label'] for key, spec in DATASET_SPECS.items()}
COLORS = {
    'underlying_load': '#2f5597',
    'net_load_with_pv': '#70ad47',
    'net_load_with_pv_battery': '#c55a11',
}
BEST_BAR_HATCH = '///'

try:
    display
except NameError:
    def display(obj):
        print(obj)

plt.rcParams.update({
    'figure.dpi': 120,
    'savefig.dpi': 300,
    'axes.grid': True,
    'grid.alpha': 0.25,
})


def save_figure(fig, filename):
    path = FIGURES_DIR / filename
    fig.savefig(path, dpi=300, bbox_inches='tight')
    print(f'Saved: {path}')


series_frames = []
metadata_rows = []
expected_datetime = None

for dataset_key, spec in DATASET_SPECS.items():
    path = DATA_DIR / spec['filename']
    df = pd.read_csv(path, parse_dates=['datetime']).sort_values('datetime').reset_index(drop=True)
    expected_columns = {'datetime', 'netload_kW'}
    missing_columns = expected_columns - set(df.columns)
    if missing_columns:
        raise ValueError(f'{path.name} is missing columns: {sorted(missing_columns)}')

    if expected_datetime is None:
        expected_datetime = df['datetime']
    elif not df['datetime'].equals(expected_datetime):
        raise ValueError(f'{path.name} does not share the same datetime index as the first dataset.')

    missing_count = int(df[['datetime', 'netload_kW']].isna().sum().sum())
    metadata_rows.append({
        'dataset': spec['label'],
        'file': spec['filename'],
        'rows': len(df),
        'start': df['datetime'].min(),
        'end': df['datetime'].max(),
        'missing_cells': missing_count,
        'households': N_HOUSEHOLDS,
        'days': N_DAYS,
        'signal_profile': SIGNAL_PROFILE,
        'definition': spec['definition'],
    })
    series_frames.append(df[['datetime', 'netload_kW']].rename(columns={'netload_kW': dataset_key}))

datasets = series_frames[0]
for frame in series_frames[1:]:
    datasets = datasets.merge(frame, on='datetime', how='inner', validate='one_to_one')

if datasets[DATASET_ORDER].isna().any().any():
    raise ValueError('The combined dataset table contains missing netload values.')

metadata = pd.DataFrame(metadata_rows)
nrmse = pd.read_csv(RESULTS_DIR / 'sa_bess_nrmse_comparison.csv')
nrmse_stddev = pd.read_csv(RESULTS_DIR / 'sa_bess_nrmse_stddev_comparison.csv')
expected_result_columns = ['model_hp', *DATASET_ORDER]

if list(nrmse.columns) != expected_result_columns:
    raise ValueError(f'nRMSE columns differ from expected columns: {list(nrmse.columns)}')
if list(nrmse_stddev.columns) != expected_result_columns:
    raise ValueError(f'nRMSE stddev columns differ from expected columns: {list(nrmse_stddev.columns)}')
if not nrmse['model_hp'].equals(nrmse_stddev['model_hp']):
    raise ValueError('nRMSE and nRMSE stddev tables do not have matching model_hp rows.')

nrmse = nrmse.set_index('model_hp')[DATASET_ORDER]
nrmse_stddev = nrmse_stddev.set_index('model_hp')[DATASET_ORDER]
metadata

## Dataset Description

Confirm the provenance, size, date range, and signal definition for each exported dataset.

In [ ]:
display(metadata)
print(f'Combined dataset shape: {datasets.shape[0]:,} timestamps x {len(DATASET_ORDER)} series')
print(f'Datetime coverage: {datasets["datetime"].min()} to {datasets["datetime"].max()}')
print(f'Expected half-hour rows per dataset: {48 * N_DAYS:,}')

## Typical Weekly Profile

Average each half-hour slot of the week across the full year to compare load-shape changes.

In [ ]:
profile_source = datasets.copy()
profile_source['week_slot'] = (
    profile_source['datetime'].dt.dayofweek * 48
    + profile_source['datetime'].dt.hour * 2
    + profile_source['datetime'].dt.minute // 30
)
typical_week = profile_source.groupby('week_slot')[DATASET_ORDER].mean().reindex(range(7 * 48))

fig, ax = plt.subplots(figsize=(13, 5))
for dataset_key in DATASET_ORDER:
    ax.plot(
        typical_week.index,
        typical_week[dataset_key],
        label=DATASET_LABELS[dataset_key],
        color=COLORS[dataset_key],
        linewidth=2.0,
    )

for day in range(8):
    ax.axvline(day * 48, color='0.85', linewidth=0.8, zorder=0)

ax.set_xticks([day * 48 + 24 for day in range(7)])
ax.set_xticklabels(['Mon', 'Tue', 'Wed', 'Thu', 'Fri', 'Sat', 'Sun'])
ax.set_ylabel('Aggregate load (kW)')
ax.set_title('Typical weekly profile of SA BESS aggregate datasets')
ax.legend(loc='upper left')
fig.tight_layout()
save_figure(fig, 'fig2_typical_week_profile.png')
plt.show()

## Distribution And Summary Statistics

Compare the annual distribution of aggregate half-hourly load across the three dataset definitions.

In [ ]:
base_stats = datasets[DATASET_ORDER].agg(['mean', 'std', 'min', 'median', 'max']).T
quantile_stats = datasets[DATASET_ORDER].quantile([0.05, 0.25, 0.75, 0.95]).T
quantile_stats.columns = ['p05', 'p25', 'p75', 'p95']
summary_stats = pd.concat([base_stats[['mean', 'std', 'min']], quantile_stats[['p05', 'p25']], base_stats[['median']], quantile_stats[['p75', 'p95']], base_stats[['max']]], axis=1)
summary_stats = summary_stats.rename(index=DATASET_LABELS).round(3)
display(summary_stats)

fig, ax = plt.subplots(figsize=(8, 5))
datasets[DATASET_ORDER].rename(columns=DATASET_LABELS).plot.box(ax=ax)
ax.set_ylabel('Aggregate load (kW)')
ax.set_title('Annual half-hourly load distributions')
ax.tick_params(axis='x', rotation=12)
fig.tight_layout()
save_figure(fig, 'fig1_dataset_distributions.png')
plt.show()

## Dataset-Level Deltas

Show how PV and PV plus battery move the typical weekly profile away from underlying load.

In [ ]:
typical_week_delta = pd.DataFrame({
    'PV effect: net load with PV minus underlying load': typical_week['net_load_with_pv'] - typical_week['underlying_load'],
    'PV and battery effect: net load with PV and battery minus underlying load': typical_week['net_load_with_pv_battery'] - typical_week['underlying_load'],
})

fig, ax = plt.subplots(figsize=(13, 5))
typical_week_delta.plot(ax=ax, linewidth=2.0)
for day in range(8):
    ax.axvline(day * 48, color='0.85', linewidth=0.8, zorder=0)
ax.axhline(0, color='0.25', linewidth=0.9)
ax.set_xticks([day * 48 + 24 for day in range(7)])
ax.set_xticklabels(['Mon', 'Tue', 'Wed', 'Thu', 'Fri', 'Sat', 'Sun'])
ax.set_ylabel('Difference from underlying load (kW)')
ax.set_title('Typical weekly dataset deltas relative to underlying load')
ax.legend(loc='lower left')
fig.tight_layout()
save_figure(fig, 'fig3_typical_week_delta.png')
plt.show()

## Model Performance By Dataset

Compare nRMSE by dataset definition. The model order follows the exported result table.

In [ ]:
display(nrmse.rename(columns=DATASET_LABELS).round(3))
display(nrmse_stddev.rename(columns=DATASET_LABELS).round(3))

plot_nrmse = nrmse.T.rename(index=DATASET_LABELS)
plot_stddev = nrmse_stddev.T
plot_stddev.index = plot_nrmse.index


def highlight_dataset_best(ax, value_table):
    bar_containers = [
        container
        for container in ax.containers
        if hasattr(container, 'patches') and len(container.patches) == len(value_table.index)
    ]
    column_to_container = dict(zip(value_table.columns, bar_containers))
    summary_rows = []

    for row_idx, dataset_label in enumerate(value_table.index):
        best_model = value_table.loc[dataset_label].idxmin()
        best_value = float(value_table.loc[dataset_label, best_model])
        patch = column_to_container[best_model].patches[row_idx]
        patch.set_hatch(BEST_BAR_HATCH)
        patch.set_edgecolor('black')
        patch.set_linewidth(1.2)
        summary_rows.append(f'{dataset_label}: {best_model} ({best_value:.2f}%)')

    return summary_rows


def add_best_summary(fig, title, rows):
    fig.text(
        0.02,
        0.02,
        title + '\n' + '\n'.join(rows),
        ha='left',
        va='bottom',
        fontsize=8.5,
        bbox={'boxstyle': 'round,pad=0.35', 'facecolor': 'white', 'edgecolor': '0.6', 'alpha': 0.95},
    )


fig, ax = plt.subplots(figsize=(14, 6))
plot_nrmse.plot(
    kind='bar',
    yerr=plot_stddev,
    ax=ax,
    width=0.82,
    capsize=2,
    error_kw={'elinewidth': 0.8, 'alpha': 0.75},
)
best_nrmse_rows = highlight_dataset_best(ax, plot_nrmse)
ax.set_ylabel('Test nRMSE (%)')
ax.set_xlabel('')
ax.set_title('PyNNLF model performance grouped by SA BESS dataset, with stddev whiskers')
ax.tick_params(axis='x', rotation=0)
handles, labels = ax.get_legend_handles_labels()
handles.append(Patch(facecolor='white', edgecolor='black', hatch=BEST_BAR_HATCH, label='Best within dataset'))
labels.append('Best within dataset')
ax.legend(
    handles,
    labels,
    title='Model/hyperparameter',
    bbox_to_anchor=(1.01, 1.0),
    loc='upper left',
    fontsize=8,
)
add_best_summary(fig, 'Best within dataset (lowest test nRMSE):', best_nrmse_rows)
fig.tight_layout(rect=(0, 0.16, 0.82, 1))
save_figure(fig, 'fig4_model_performance_by_dataset.png')
plt.show()

fig, ax = plt.subplots(figsize=(14, 6))
plot_stddev.plot(
    kind='bar',
    ax=ax,
    width=0.82,
)
best_stddev_rows = highlight_dataset_best(ax, plot_stddev)
ax.set_ylabel('Test nRMSE stddev (%)')
ax.set_xlabel('')
ax.set_title('PyNNLF test nRMSE stddev grouped by SA BESS dataset')
ax.tick_params(axis='x', rotation=0)
handles, labels = ax.get_legend_handles_labels()
handles.append(Patch(facecolor='white', edgecolor='black', hatch=BEST_BAR_HATCH, label='Lowest stddev within dataset'))
labels.append('Lowest stddev within dataset')
ax.legend(
    handles,
    labels,
    title='Model/hyperparameter',
    bbox_to_anchor=(1.01, 1.0),
    loc='upper left',
    fontsize=8,
)
add_best_summary(fig, 'Lowest variability within dataset (test nRMSE stddev):', best_stddev_rows)
fig.tight_layout(rect=(0, 0.16, 0.82, 1))
save_figure(fig, 'fig5_test_nrmse_stddev_by_dataset.png')
plt.show()

## Figure Outputs

The notebook writes PNG figures to `publication/journal_article_1/results/figures`.

In [ ]:
for path in sorted(FIGURES_DIR.glob('fig*.png')):
    print(path)